# 01 - Data Audit

## هدف
در این نوت‌بوک، داده‌های تصادفات جاده‌ای بریتانیا و فرانسه از نظر ساختار، کیفیت، مقادیر گمشده، رکوردهای تکراری، متغیر هدف و قابلیت تطبیق بین‌کشوری بررسی می‌شوند.

در این مرحله هیچ پاک‌سازی پیچیده یا مهندسی ویژگی انجام نمی‌شود. هدف فقط تصمیم‌گیری درباره مناسب بودن داده‌ها برای ادامه پژوهش است

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# UK file paths
uk_collisions_path = "../data/raw/UK/dft-road-casualty-statistics-collision-2025.csv"
uk_vehicles_path = "../data/raw/UK/dft-road-casualty-statistics-vehicle-2025.csv"
uk_casualties_path = "../data/raw/UK/dft-road-casualty-statistics-casualty-2025.csv"
uk_guide_path = "../data/raw/UK/dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx"

# France file paths
fr_characteristics_path = "../data/raw/France/caract-2024.csv"
fr_locations_path = "../data/raw/France/lieux-2024.csv"
fr_vehicles_path = "../data/raw/France/vehicules-2024.csv"
fr_users_path = "../data/raw/France/usagers-2024.csv"

In [ ]:
uk_collisions = pd.read_csv(uk_collisions_path)
uk_vehicles = pd.read_csv(uk_vehicles_path)
uk_casualties = pd.read_csv(uk_casualties_path)

print("UK Collisions:", uk_collisions.shape)
print("UK Vehicles:", uk_vehicles.shape)
print("UK Casualties:", uk_casualties.shape)

In [ ]:
fr_characteristics = pd.read_csv(fr_characteristics_path, sep=";")
fr_locations = pd.read_csv(fr_locations_path, sep=";")
fr_vehicles = pd.read_csv(fr_vehicles_path, sep=";")
fr_users = pd.read_csv(fr_users_path, sep=";")

print("France Characteristics:", fr_characteristics.shape)
print("France Locations:", fr_locations.shape)
print("France Vehicles:", fr_vehicles.shape)
print("France Users:", fr_users.shape)

## 1. بررسی اولیه ساختار و کیفیت داده‌ها

در این بخش، اندازه جداول، مقادیر گمشده و رکوردهای تکراری کامل در داده‌های خام بریتانیا و فرانسه بررسی می‌شوند. در این مرحله هیچ رکوردی حذف یا اصلاح نمی‌شود.

In [ ]:
def basic_audit(df, name):
    return {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing_Values": df.isna().sum().sum(),
        "Duplicate_Rows": df.duplicated().sum()
    }

In [ ]:
audit_results = [
    basic_audit(uk_collisions, "UK Collisions"),
    basic_audit(uk_vehicles, "UK Vehicles"),
    basic_audit(uk_casualties, "UK Casualties"),
    basic_audit(fr_characteristics, "France Characteristics"),
    basic_audit(fr_locations, "France Locations"),
    basic_audit(fr_vehicles, "France Vehicles"),
    basic_audit(fr_users, "France Users")
]

audit_summary = pd.DataFrame(audit_results)

audit_summary

## 2. بررسی مقادیر گمشده در سطح متغیرها

در این بخش، توزیع مقادیر گمشده در ستون‌های هر جدول بررسی می‌شود تا مشخص شود آیا مقادیر گمشده در متغیرهای مهم پژوهش متمرکز هستند یا عمدتاً به متغیرهای فرعی مربوط می‌شوند.

In [ ]:
uk_missing = uk_collisions.isna().sum()

uk_missing[uk_missing > 0].sort_values(ascending=False)

In [ ]:
fr_characteristics_missing = fr_characteristics.isna().sum()
fr_locations_missing = fr_locations.isna().sum()

print("France Characteristics:")
display(
    fr_characteristics_missing[
        fr_characteristics_missing > 0
    ].sort_values(ascending=False)
)

print("\nFrance Locations:")
display(
    fr_locations_missing[
        fr_locations_missing > 0
    ].sort_values(ascending=False)
)

## 3. بررسی متغیر هدف شدت تصادف

در این بخش، ساختار و توزیع متغیر شدت در داده‌های بریتانیا و فرانسه بررسی می‌شود. هدف، ارزیابی امکان تعریف یک متغیر هدف مشترک و قابل دفاع برای تحلیل و اعتبارسنجی بین‌کشوری است

In [ ]:
uk_collisions["collision_severity"].value_counts().sort_index()

In [ ]:
fr_users["grav"].value_counts().sort_index()

### 3.1 تعریف شدت تصادف در دو مجموعه‌داده

در داده‌های بریتانیا، شدت تصادف در سطح حادثه و براساس شدیدترین آسیب ثبت‌شده در میان مصدومان آن تصادف تعریف شده است.

در داده‌های فرانسه، شدت (`grav`) در سطح افراد درگیر ثبت شده است. بنابراین برای ایجاد متغیر شدت در سطح تصادف، شدیدترین پیامد ثبت‌شده در میان افراد هر تصادف استخراج می‌شود.

کدهای شدت فرانسه:
- 1: بدون آسیب (Uninjured)
- 2: فوت‌شده (Killed)
- 3: مصدوم بستری (Hospitalised injured)
- 4: مصدوم با آسیب خفیف (Slightly injured)

In [ ]:
severity_rank_fr = {
    1: 0,  # Uninjured
    4: 1,  # Slight
    3: 2,  # Hospitalised
    2: 3   # Killed
}

fr_users["severity_rank"] = fr_users["grav"].map(severity_rank_fr)

In [ ]:
fr_collision_severity = (
    fr_users.groupby("Num_Acc")["severity_rank"]
    .max()
    .reset_index()
)

fr_collision_severity["severity_rank"].value_counts().sort_index()

In [ ]:
uk_severity_map = {
    1: "Fatal",
    2: "Serious",
    3: "Slight"
}

uk_collisions["severity_class"] = (
    uk_collisions["collision_severity"]
    .map(uk_severity_map)
)

uk_collisions["severity_class"].value_counts()

In [ ]:
fr_severity_map = {
    1: "Slight",
    2: "Serious",
    3: "Fatal"
}

fr_collision_severity["severity_class"] = (
    fr_collision_severity["severity_rank"]
    .map(fr_severity_map)
)

fr_collision_severity["severity_class"].value_counts()

In [ ]:
severity_comparison = pd.DataFrame({
    "UK_Count": uk_collisions["severity_class"].value_counts(),
    "France_Count": fr_collision_severity["severity_class"].value_counts()
})

severity_comparison["UK_Percent"] = (
    severity_comparison["UK_Count"] / len(uk_collisions) * 100
)

severity_comparison["France_Percent"] = (
    severity_comparison["France_Count"] / len(fr_collision_severity) * 100
)

severity_comparison

In [ ]:
severity_comparison.to_csv(
    "../outputs/tables/severity_distribution_uk_france.csv",
    index=True
)

## 4. بررسی سازگاری ویژگی‌ها بین بریتانیا و فرانسه

در این بخش، متغیرهای موجود در مجموعه‌داده‌های بریتانیا و فرانسه بررسی می‌شوند تا ویژگی‌هایی شناسایی شوند که از نظر مفهومی در هر دو کشور قابل مقایسه باشند.

در این مرحله هیچ ادغام، بازکدگذاری یا مهندسی ویژگی انجام نمی‌شود. هدف، تعیین یک مجموعه ویژگی مشترک، ساده و قابل دفاع برای اعتبارسنجی بین‌کشوری است

In [ ]:
print("UK Collisions columns:")
print(uk_collisions.columns.tolist())

In [ ]:
print("UK Vehicles columns:")
print(uk_vehicles.columns.tolist())

In [ ]:
print("France Characteristics columns:")
print(fr_characteristics.columns.tolist())

In [ ]:
print("France Locations columns:")
print(fr_locations.columns.tolist())

In [ ]:
print("France Vehicles columns:")
print(fr_vehicles.columns.tolist())

### 4.1 بررسی متغیرهای مشترک اصلی

در مرحله نخست، پنج متغیر محیطی و جاده‌ای که در هر دو مجموعه‌داده دارای معادل مفهومی مستقیم هستند بررسی می‌شوند:

- شرایط آب‌وهوا
- شرایط روشنایی
- وضعیت سطح جاده
- محدودیت سرعت
- محیط شهری/غیرشهری

هدف، بررسی امکان هماهنگ‌سازی این متغیرها بدون نیاز به تبدیل‌های پیچیده است.

In [ ]:
uk_features_check = [
    "weather_conditions",
    "light_conditions",
    "road_surface_conditions",
    "speed_limit",
    "urban_or_rural_area"
]

for col in uk_features_check:
    print(f"\n--- {col} ---")
    print(uk_collisions[col].value_counts(dropna=False).sort_index())

In [ ]:
fr_features_check = [
    "atm",
    "lum",
    "surf",
    "vma",
    "agg"
]

for col in fr_features_check:
    print(f"\n--- {col} ---")
    print(fr_characteristics[col].value_counts(dropna=False).sort_index()
          if col in fr_characteristics.columns
          else fr_locations[col].value_counts(dropna=False).sort_index())

### 4.2 بررسی محدودیت سرعت در داده فرانسه

متغیر `vma` در داده فرانسه دارای مقادیر متنوعی است. پیش از هرگونه پاک‌سازی یا هماهنگ‌سازی با داده بریتانیا، فراوانی مقادیر نامشخص و غیرمعمول در سطح تصادف بررسی می‌شود.

In [ ]:
# بررسی مقادیر غیرمعمول سرعت مجاز در France

unusual_vma = fr_locations[
    (fr_locations["vma"] == -1) |
    (fr_locations["vma"] > 130)
]

print("Rows with unusual/unknown VMA:", len(unusual_vma))
print("Unique accidents affected:", unusual_vma["Num_Acc"].nunique())
print("Percentage of France accidents affected:",
      unusual_vma["Num_Acc"].nunique() / fr_locations["Num_Acc"].nunique() * 100)

In [ ]:
fr_locations.loc[
    fr_locations["vma"] > 130,
    ["Num_Acc", "vma"]
].sort_values("vma")

**نتیجه بررسی `vma`:**

در داده فرانسه، 3,659 ردیف دارای مقدار نامشخص (`-1`) یا مقدار بالاتر از 130 در متغیر محدودیت سرعت (`vma`) بودند که 3,548 تصادف یکتا (حدود 6.52٪ از تصادفات) را تحت تأثیر قرار می‌دهند. مقادیر بالاتر از 130 بسیار محدود هستند و تصمیم درباره نحوه مدیریت مقادیر نامعتبر یا نامشخص به مرحله آماده‌سازی داده موکول می‌شود. در مرحله ممیزی هیچ مقداری اصلاح یا حذف نشد.

In [ ]:
print("--- France hrmn ---")
print(fr_characteristics["hrmn"].head(10))
print("Missing:", fr_characteristics["hrmn"].isna().sum())

print("\n--- France int ---")
print(fr_characteristics["int"].value_counts(dropna=False).sort_index())

print("\n--- France catr ---")
print(fr_locations["catr"].value_counts(dropna=False).sort_index())

### 4.3 بررسی زمان، تقاطع و نوع جاده

در این بخش، ساختار متغیرهای زمان وقوع تصادف، نوع تقاطع و نوع/رده جاده در دو مجموعه‌داده بررسی می‌شود تا امکان استفاده از آن‌ها در فضای ویژگی مشترک ارزیابی شود. در این مرحله هیچ بازکدگذاری یا مهندسی ویژگی انجام نمی‌شود.

In [ ]:
print("--- UK time ---")
print(uk_collisions["time"].head(10))
print("Missing:", uk_collisions["time"].isna().sum())

print("\n--- UK junction_detail ---")
print(
    uk_collisions["junction_detail"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n--- UK road_type ---")
print(
    uk_collisions["road_type"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n--- UK first_road_class ---")
print(
    uk_collisions["first_road_class"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
print("--- France hrmn ---")
print(fr_characteristics["hrmn"].head(10))
print("Missing:", fr_characteristics["hrmn"].isna().sum())

print("\n--- France int ---")
print(
    fr_characteristics["int"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n--- France catr ---")
print(
    fr_locations["catr"]
    .value_counts(dropna=False)
    .sort_index()
)

### 4.4 جمع‌بندی سازگاری اولیه ویژگی‌های مشترک

براساس بررسی ساختار داده، توزیع مقادیر و مستندات رسمی دو مجموعه‌داده، ویژگی‌های زیر به‌عنوان نامزدهای اولیه برای ایجاد فضای ویژگی مشترک بین بریتانیا و فرانسه انتخاب شدند.

In [ ]:
feature_compatibility = pd.DataFrame({
    "Concept": [
        "Time",
        "Light Conditions",
        "Weather Conditions",
        "Road Surface",
        "Urban/Rural",
        "Speed Limit",
        "Junction",
        "Road Class"
    ],

    "Persian_Name": [
        "زمان وقوع تصادف",
        "شرایط روشنایی",
        "شرایط آب‌وهوایی",
        "وضعیت سطح جاده",
        "شهری یا برون‌شهری بودن",
        "حداکثر سرعت مجاز",
        "وضعیت/نوع تقاطع",
        "رده جاده"
    ],

    "UK_Field": [
        "time",
        "light_conditions",
        "weather_conditions",
        "road_surface_conditions",
        "urban_or_rural_area",
        "speed_limit",
        "junction_detail",
        "first_road_class"
    ],

    "France_Field": [
        "hrmn",
        "lum",
        "atm",
        "surf",
        "agg",
        "vma",
        "int",
        "catr"
    ],

    "Compatibility": [
        "High",
        "High",
        "High",
        "High",
        "High",
        "Moderate-High",
        "Moderate-High",
        "Moderate"
    ],

    "Decision": [
        "Keep",
        "Keep",
        "Keep",
        "Keep",
        "Keep",
        "Keep - cleaning required",
        "Keep - harmonization required",
        "Review before final inclusion"
    ]
})

feature_compatibility

In [ ]:
feature_compatibility.to_csv(
    "../outputs/tables/feature_compatibility_uk_france.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Feature compatibility table saved.")

In [ ]:
print("UK Collisions")
print("Rows:", len(uk_collisions))
print("Unique collision IDs:", uk_collisions["collision_index"].nunique())

print("\nUK Vehicles")
print("Rows:", len(uk_vehicles))
print("Unique collision IDs:", uk_vehicles["collision_index"].nunique())

print("\nUK Casualties")
print("Rows:", len(uk_casualties))
print("Unique collision IDs:", uk_casualties["collision_index"].nunique())

In [ ]:
print("France Characteristics")
print("Rows:", len(fr_characteristics))
print("Unique Num_Acc:", fr_characteristics["Num_Acc"].nunique())

print("\nFrance Locations")
print("Rows:", len(fr_locations))
print("Unique Num_Acc:", fr_locations["Num_Acc"].nunique())

print("\nFrance Vehicles")
print("Rows:", len(fr_vehicles))
print("Unique Num_Acc:", fr_vehicles["Num_Acc"].nunique())

print("\nFrance Users")
print("Rows:", len(fr_users))
print("Unique Num_Acc:", fr_users["Num_Acc"].nunique())